# Cartographie des Points Noirs — Réseau Ferroviaire Belge
 
**Source :** Open Data Infrabel  
**Stack :** Python · Pandas · Folium · Plotly  

---
## Plan du notebook
1. Installation des dépendances
2. Téléchargement des données Infrabel
3. Exploration des données brutes
4. Nettoyage et transformation
5. Agrégation par gare
6. Cartographie interactive (Folium)
7. Analyse graphique (Plotly)
8. Conclusions & insights

---
##  Étape 0 — Installation des dépendances

In [1]:
# Installation des librairies nécessaires
# À exécuter une seule fois
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'pandas', 'folium', 'plotly', 'requests', 'openpyxl',
                '--quiet'])
print('✅ Librairies installées')

✅ Librairies installées


---
## Imports

In [2]:
import requests
import pandas as pd
import folium
from folium.plugins import HeatMap
import plotly.express as px
import plotly.graph_objects as go
from io import StringIO
import os
import warnings
warnings.filterwarnings('ignore')

# Dossiers de sortie
os.makedirs('data/raw',   exist_ok=True)
os.makedirs('data/clean', exist_ok=True)

print('✅ Imports OK')

✅ Imports OK


---
## 1️ Téléchargement des données Infrabel

In [3]:
# --- Récupération de l'index des fichiers mensuels ---
INDEX_URL = (
    'https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/'
    'stiptheid-gegevens-maandelijksebestanden/records?limit=100&lang=fr'
)

print('Récupération de l\'index...')
resp = requests.get(INDEX_URL, timeout=30)
df_index = pd.DataFrame(resp.json()['results'])
df_index = df_index.sort_values('mois', ascending=False).reset_index(drop=True)

print(f'    {len(df_index)} mois disponibles')
print(f'    Période : {df_index["mois"].iloc[-1]} → {df_index["mois"].iloc[0]}')
df_index.head()

Récupération de l'index...
    100 mois disponibles
    Période : 2014-03 → 2026-02


,mois,link_to_data
0,2026-02,https://fr.ftp.opendatasoft.com/infrabel/Punct...
1,2026-01,https://fr.ftp.opendatasoft.com/infrabel/Punct...
2,2025-12,https://fr.ftp.opendatasoft.com/infrabel/Punct...
3,2025-11,https://fr.ftp.opendatasoft.com/infrabel/Punct...
4,2025-10,https://fr.ftp.opendatasoft.com/infrabel/Punct...


In [ ]:
# --- Téléchargement des 12 derniers mois ---
NB_MOIS = 12
df_recent = df_index.head(NB_MOIS)

print(f'🚄 Téléchargement des {NB_MOIS} derniers mois...')
print(f'   Période : {df_recent["mois"].iloc[-1]} → {df_recent["mois"].iloc[0]}\n')

all_dfs = []

for _, row in df_recent.iterrows():
    mois = row['mois']
    url  = row['link_to_data']
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        df_month = pd.read_csv(StringIO(r.text), sep=',', encoding='utf-8')
        df_month['mois'] = mois
        all_dfs.append(df_month)
        print(f'   ✅ {mois} — {len(df_month):,} lignes | colonnes : {list(df_month.columns)[:4]}...')
    except Exception as e:
        print(f'   ⚠️  {mois} — Erreur : {e}')

# Fusion de tous les mois
df_raw = pd.concat(all_dfs, ignore_index=True)

# Sauvegarde brute
df_raw.to_csv('../data/raw/ponctualite_merged.csv', index=False, encoding='utf-8-sig')

print(f'\n Total : {len(df_raw):,} lignes | {len(df_raw.columns)} colonnes')
print(f'Sauvegardé : data/raw/ponctualite_merged.csv')

🚄 Téléchargement des 12 derniers mois...
   Période : 2025-01 → 2026-02

   ✅ 2026-02 — 1,897,533 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...


---
## 2️ Exploration des données brutes

In [ ]:
# Structure générale
print(f'Dimensions : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')
print(f'\nColonnes :')
for col in df_raw.columns:
    print(f'  - {col} ({df_raw[col].dtype}) | ex: {df_raw[col].iloc[0]}')

In [ ]:
display(df_raw.head())
df_raw.info()

In [ ]:
# Valeurs manquantes

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({'Manquants': missing, 'Pct (%)': missing_pct})

In [ ]:
# Aperçu des données
df_raw.head(10)

In [ ]:
df_raw['CIRC_TYP'].unique()

In [ ]:
df_raw.drop(columns=['CIRC_TYP'], inplace=True)

In [ ]:
df_raw['OP1_COD'].unique()

In [ ]:
df_raw['THOP1_COD'].unique()

In [ ]:
# Comparaison directe
(df_raw['OP1_COD'] == df_raw['THOP1_COD']).value_counts(dropna=False)

In [ ]:
# Voir quelques lignes où ils diffèrent, avec le contexte
df_raw[df_raw['OP1_COD'] != df_raw['THOP1_COD']][
    ['RELATION', 'PTCAR_LG_NM_NL', 'OP1_COD', 'THOP1_COD', 'DELAY_DEP']
].head(20)

In [ ]:
# convertir les veleurs en minutes
(df_raw['DELAY_DEP'] / 60).describe()


In [ ]:
# Nettoyer les outliers et garder que les lignes avec un retard valide
df_valid = df_raw[
    (df_raw['DELAY_DEP'].notna()) &
    (df_raw['DELAY_DEP'] >= -30) &      # max 30min en avance (généreux)
    (df_raw['DELAY_DEP'] <= 1800)        # max 6h de retard
].copy()

df_valid['delay_min'] = df_valid['DELAY_DEP'] / 60

print(f'Lignes gardées : {len(df_valid):,} / {len(df_raw):,}')
display(df_valid['delay_min'].describe())

df_valid['delay_min'] = df_valid['DELAY_DEP'] / 60


---
## 3️ Nettoyage et transformation

In [ ]:
df = df_raw.copy()

# --- Affichage des colonnes disponibles pour adapter le nettoyage ---
print('Colonnes disponibles :', list(df.columns))

# --- Normalisation des noms de colonnes (minuscules, sans espaces) ---
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
)
print('\nColonnes normalisées :', list(df.columns))

In [ ]:
df.rename(columns={
    'ptcar_lg_nm_nl': 'gare',
    'delay_dep': 'retard',
    'mois': 'date'
}, inplace=True)

df.head()  # → maintenant les colonnes sont 'gare', 'retard', 'date'

In [ ]:
# Colonnes de date à convertir en datetime
date_cols = ['datdep', 'real_date_arr', 'real_date_dep', 'planned_date_arr', 'planned_date_dep', 'date']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%d%b%Y', errors='coerce')

print('✅ Dates converties')
df[date_cols].dtypes


In [ ]:
df[date_cols]

In [ ]:
# 1. Doublons
avant = len(df)
df = df.drop_duplicates()
print(f'Doublons supprimés : {avant - len(df):,}')

# 2. Nettoyer noms de gares
df['gare'] = df['gare'].astype(str).str.strip().str.title()
df = df[df['gare'] != 'Nan']

print(f'✅ {len(df):,} lignes propres')

In [ ]:
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
cols_to_drop = [
    'train_no', 'train_serv', 'op1_cod', 'thop1_cod',
    'real_date_arr', 'real_date_dep', 'planned_date_arr', 'planned_date_dep',
    'real_time_arr', 'real_time_dep', 'planned_time_arr', 'planned_time_dep',
    'line_no_arr', 'relation_direction'
]

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f'Colonnes restantes : {df.shape[1]} → {list(df.columns)}')
print(f'Mémoire : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
# optimise les types
for col in ['relation', 'gare', 'line_no_dep']:
    df[col] = df[col].astype('category')

print(f'Mémoire après catégorisation : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

In [ ]:
df.info()

In [ ]:
# Filtre outliers
df = df[
    (df['retard'].notna()) &
    (df['retard'] >= -30) &
    (df['retard'] <= 1800)
].copy()

# Retard en minutes
df['retard_min'] = (df['retard'] / 60).round(2)

print(f'Lignes finales : {len(df):,}')
print(f'Mémoire finale : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

In [ ]:

# Sauvegarde
import os
os.makedirs('data/clean', exist_ok=True)

df.to_parquet('data/clean/ponctualite_clean.parquet', index=False)

taille = os.path.getsize('data/clean/ponctualite_clean.parquet') / 1e6
print(f'Lignes sauvegardées : {len(df):,}')
print(f'Taille sur disque   : {taille:.1f} MB')
print(f'Colonnes            : {list(df.columns)}')

---
## 4️ KPI's blackspots

In [ ]:
# Agrégation par gare — création du DataFrame blackspots
blackspots = df.groupby('gare').agg(
    passages        = ('retard_min', 'count'),
    retard_moyen    = ('retard_min', 'mean'),
    pct_en_retard   = ('retard_min', lambda x: (x > 5).mean() * 100),
    minutes_perdues = ('retard_min', lambda x: x[x > 0].sum())
).round(2).reset_index()

print(f'Gares identifiées : {len(blackspots)}')
display(blackspots.head(10))

In [ ]:
# Garder uniquement les gares avec assez de passages (vraies gares voyageurs)
blackspots = blackspots[blackspots['passages'] >= 5000].copy()
blackspots = blackspots.sort_values('minutes_perdues', ascending=False).reset_index(drop=True)

print(f'Gares significatives : {len(blackspots)}')
display(blackspots.head(15))

---
## 5️ Coordonnées GPS des gares

In [ ]:
import requests

#  1. Récupérer le référentiel iRail ─────────────────────────
url = "https://api.irail.be/stations/?format=json&lang=nl"
stations_ref = pd.DataFrame([{
    'gare_key' : s['name'].strip().upper(),
    'lat'      : float(s['locationY']),
    'lon'      : float(s['locationX'])
} for s in requests.get(url).json()['station']])

#  2. Clé de jointure sur blackspots ─────────────────────────
blackspots['gare_key'] = blackspots['gare'].str.upper().str.strip()

#  3. Toutes les corrections FR → NL ─────────────────────────
corrections = {
    'LIEGE-GUILLEMINS'    : 'LUIK-GUILLEMINS',
    'LIEGE-CARRE'         : 'LUIK-CARRE',
    'LIEGE-SAINT-LAMBERT' : 'LUIK-SINT-LAMBERTUS',
    'NAMUR'               : 'NAMEN',
    'CHARLEROI-CENTRAL'   : 'CHARLEROI-CENTRAAL',
    'MONS'                : 'BERGEN',
    'TUBIZE'              : 'TUBEKE',
    "BRAINE-L'ALLEUD"     : 'EIGENBRAKEL',
    'BRAINE-LE-COMTE'     : "'S GRAVENBRAKEL",
    'JURBISE'             : 'JURBEKE',
    'LA HULPE'            : 'TERHULPEN',
    'LOKEREN-OOST'        : 'LOKEREN',
    'SINT-NIKLAAS-OOST'   : 'SINT-NIKLAAS',
    'NIVELLES'            : 'NIJVEL',
    'BEVEREN(WAAS)'       : 'BEVEREN',
}
blackspots['gare_key'] = blackspots['gare_key'].replace(corrections)

#  4. Jointure ────────────────────────────────────────────────
bs_geo = blackspots.merge(stations_ref, on='gare_key', how='left')

#  5. Résultats ───────────────────────────────────────────────
matched   = bs_geo['lat'].notna().sum()
unmatched = bs_geo['lat'].isna().sum()
print(f'Avec coordonnées : {matched} / {len(bs_geo)}')
print(f'Sans coordonnées : {unmatched}')

if unmatched > 0:
    reste = bs_geo[bs_geo['lat'].isna()][['gare','passages']]\
            .sort_values('passages', ascending=False)
    print(f'\nNon-matchés (gares fret/techniques, à ignorer) :')
    print(reste.head(10))

In [ ]:
for terme in ['GRAVENBRAKEL', 'ALLIANCE', 'LOUVIERE', 'HERBAT', 
              'JURBEKE', 'TERHULPEN', 'HENNE', 'LOKEREN', 'HULPE']:
    matches = stations_ref[stations_ref['gare_key'].str.contains(terme, na=False)]
    if len(matches) > 0:
        print(f"{terme} → {matches['gare_key'].tolist()}")
    else:
        print(f"{terme} → ❌ absent d'iRail")

---
## 6️ Cartographie interactive (Folium)

In [ ]:
import folium
from folium.plugins import HeatMap
import os

os.makedirs('outputs', exist_ok=True)

#  Rang sur bs_geo ───────────────────────────────────────────
bs_geo = bs_geo.sort_values('minutes_perdues', ascending=False).reset_index(drop=True)
bs_geo['rang'] = range(1, len(bs_geo) + 1)

#  Données avec coordonnées ──────────────────────────────────
df_map = bs_geo[bs_geo['lat'].notna()].copy()
df_map['intensite'] = df_map['minutes_perdues'] / df_map['minutes_perdues'].max()
heat_data = df_map[['lat', 'lon', 'intensite']].values.tolist()

#  Carte ─────────────────────────────────────────────────────
carte = folium.Map(location=[50.5, 4.5], zoom_start=9, tiles='CartoDB positron')

#  Titre ─────────────────────────────────────────────────────
titre = '''
<div style="position:fixed; top:15px; left:60px; z-index:1000;
     background:white; padding:10px 15px; border-radius:6px;
     border-left:4px solid #bd0026; font-family:Arial;
     box-shadow:2px 2px 6px rgba(0,0,0,0.2)">
    <b style="font-size:14px; color:#1a2940">Blackspots Infrabel</b><br>
    <span style="font-size:11px; color:#555">Minutes perdues par gare · 2025-2026</span>
</div>'''
carte.get_root().html.add_child(folium.Element(titre))

#  HeatMap ───────────────────────────────────────────────────
HeatMap(
    heat_data,
    radius=18,
    blur=15,
    max_zoom=10,
    gradient={0.2: '#ffffb2', 0.5: '#fd8d3c', 0.8: '#f03b20', 1.0: '#bd0026'}
).add_to(carte)

#  Marqueurs Top 15 ──────────────────────────────────────────
top15 = df_map.nlargest(15, 'minutes_perdues')

for _, row in top15.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color='#1a2940',
        fill=True,
        fill_color='#e74c3c',
        fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{row['gare']}</b><br>"
            f"Rang : #{int(row['rang'])}<br>"
            f"Minutes perdues : {row['minutes_perdues']:,.0f}<br>"
            f"Retard moyen : {row['retard_moyen']:.1f} min<br>"
            f"% en retard : {row['pct_en_retard']:.1f}%",
            max_width=220
        ),
        tooltip=f"#{int(row['rang'])} {row['gare']}"
    ).add_to(carte)

#   Sauvegarde ────────────────────────────────────────────────
carte.save('outputs/blackspots_heatmap.html')
print('Carte sauvegardée : outputs/blackspots_heatmap.html')
carte

---
## 7️ Analyse graphique (Plotly)

In [ ]:
# Analyse graphique (Plotly)

# Reconstruire la tendance depuis datdep (fiable) plutôt que date
tendance = df.copy()
tendance['mois'] = tendance['datdep'].dt.to_period('M').astype(str)

tendance = tendance.groupby('mois').agg(
    retard_moyen = ('retard_min', 'mean'),
    pct_en_retard = ('retard_min', lambda x: (x > 5).mean() * 100),
    total_passages = ('retard_min', 'count')
).round(2).reset_index()

tendance['mois'] = pd.to_datetime(tendance['mois'])
tendance = tendance.sort_values('mois')

print(f"Nombre de mois : {len(tendance)}")
print(tendance)
print(tendance.columns.tolist())


In [ ]:

# Graphique
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig3 = make_subplots(specs=[[{'secondary_y': True}]])

fig3.add_trace(go.Bar(
    x=tendance['mois'],
    y=tendance['total_passages'],
    name='Passages',
    marker_color='#d6e4f0',
    opacity=0.6
), secondary_y=True)

fig3.add_trace(go.Scatter(
    x=tendance['mois'],
    y=tendance['pct_en_retard'],
    name='% en retard',
    mode='lines+markers',
    line=dict(color='#bd0026', width=2),
    marker=dict(size=7)
), secondary_y=False)

fig3.add_trace(go.Scatter(
    x=tendance['mois'],
    y=tendance['retard_moyen'],
    name='Retard moyen (min)',
    mode='lines+markers',
    line=dict(color='#fd8d3c', width=2, dash='dot'),
    marker=dict(size=7)
), secondary_y=False)

fig3.update_layout(
    title='Évolution mensuelle de la ponctualité — Réseau Infrabel 2025-2026',
    height=450,
    font_family='Arial',
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=14,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    xaxis=dict(
        tickformat='%b %Y',
        showgrid=True,
        gridcolor='#f0f0f0'
    )
)
fig3.update_yaxes(title_text='% en retard / Retard moyen (min)', secondary_y=False)
fig3.update_yaxes(title_text='Nombre de passages', secondary_y=True, showgrid=False)

os.makedirs('outputs', exist_ok=True)
fig3.write_html('outputs/tendance_mensuelle.html')
fig3.show()


---
## 8️ Conclusions & Insights

> 

###  Observations clés
- **Gare la plus problématique :** ...
- **Retard moyen global :** ...
- **Tendance :** ...
A suivre
